# M2: FM100 group comparisons

`PLANScores.md` M2: `comparisons.compare_fm100_feature` (Mann-Whitney U + effect size, subject-pooled scores, same convention `beh/scripts/features.py`'s `compare_shape_feature` uses) across `comparisons.FEATURES` (TES, PES_RG, PES_BY, VKS_MajRad, VKS_MinRad, VKS_Angle) for the four requested pairs: CTR vs PD, HC vs protan, HC vs deutan, protan vs deutan.

`VKS_Angle` is periodic (folds to `[0, 180)`) -- compared with the same folded-Mann-Whitney approximation `beh/`'s `orientation_deg` already uses (valid as long as a group's angles don't straddle the 0/180 wrap point; see `comparisons.py`'s module docstring).

In [1]:
import sys
sys.path.append('../scripts')

import pandas as pd

import comparisons
import loader

df = loader.load_fm100_raw()

pairs = [
    ('CTR', None, 'PD', None),
    ('CTR', None, 'CVD', 'protan'),
    ('CTR', None, 'CVD', 'deutan'),
    ('CVD', 'protan', 'CVD', 'deutan'),
]
pair_labels = ['CTR vs PD', 'HC vs protan', 'HC vs deutan', 'protan vs deutan']

In [2]:
rows = []
for (g1, sg1, g2, sg2), label in zip(pairs, pair_labels):
    for feature in comparisons.FEATURES:
        result = comparisons.compare_fm100_feature(df, feature, group1=g1, subgroup1=sg1, group2=g2, subgroup2=sg2)
        rows.append({'comparison': label, **result})
results = pd.DataFrame(rows)
results

,comparison,feature,u_val,p_value,rbc,cles,n1,n2
0,CTR vs PD,TES,31.5,0.022133,-0.611111,0.194444,27,6
1,CTR vs PD,PES_RG,26.5,0.011698,-0.672840,0.163580,27,6
2,CTR vs PD,PES_BY,33.0,0.026603,-0.592593,0.203704,27,6
3,CTR vs PD,VKS_MajRad,33.0,0.023711,-0.592593,0.203704,27,6
4,CTR vs PD,VKS_MinRad,33.0,0.023711,-0.592593,0.203704,27,6
5,CTR vs PD,VKS_Angle,82.0,0.981874,0.012346,0.506173,27,6
6,HC vs protan,TES,5.0,0.000022,-0.958848,0.020576,27,9
7,HC vs protan,PES_RG,0.0,0.000010,-1.000000,0.000000,27,9
8,HC vs protan,PES_BY,53.0,0.012977,-0.563786,0.218107,27,9
9,HC vs protan,VKS_MajRad,7.0,0.000031,-0.942387,0.028807,27,9


## Just the significant rows (p < 0.05, uncorrected -- matching every other group-comparison battery in this project, e.g. `beh/`'s M1)

In [3]:
results[results['p_value'] < 0.05].sort_values('p_value')

,comparison,feature,u_val,p_value,rbc,cles,n1,n2
16,HC vs deutan,VKS_MinRad,7.0,0.000004,-0.935185,0.032407,27,8
15,HC vs deutan,VKS_MajRad,8.0,0.000006,-0.925926,0.037037,27,8
7,HC vs protan,PES_RG,0.0,0.000010,-1.000000,0.000000,27,9
6,HC vs protan,TES,5.0,0.000022,-0.958848,0.020576,27,9
13,HC vs deutan,PES_RG,0.0,0.000024,-1.000000,0.000000,27,8
9,HC vs protan,VKS_MajRad,7.0,0.000031,-0.942387,0.028807,27,9
12,HC vs deutan,TES,7.5,0.000085,-0.930556,0.034722,27,8
10,HC vs protan,VKS_MinRad,16.0,0.000125,-0.868313,0.065844,27,9
14,HC vs deutan,PES_BY,26.0,0.001365,-0.759259,0.120370,27,8
11,HC vs protan,VKS_Angle,201.0,0.003900,0.654321,0.827160,27,9


## Holm-corrected, per pair (M3)

`comparisons.correct_multiple_comparisons` (Holm, the family-wise-error
default), applied once per comparison (6 features = one family), matching
how `ssvepBeh/`'s own correlation battery is corrected. Family-wise error
control is more conservative than the uncorrected view above -- some rows
significant there don't survive here.

In [4]:
corrected = pd.concat([comparisons.correct_multiple_comparisons(results[results['comparison'] == label]) for label in pair_labels])
corrected[corrected['significant']].sort_values('p_corrected')

,comparison,feature,u_val,p_value,rbc,cles,n1,n2,p_corrected,significant
16,HC vs deutan,VKS_MinRad,7.0,0.000004,-0.935185,0.032407,27,8,0.000023,True
15,HC vs deutan,VKS_MajRad,8.0,0.000006,-0.925926,0.037037,27,8,0.000028,True
7,HC vs protan,PES_RG,0.0,0.000010,-1.000000,0.000000,27,9,0.000059,True
13,HC vs deutan,PES_RG,0.0,0.000024,-1.000000,0.000000,27,8,0.000096,True
6,HC vs protan,TES,5.0,0.000022,-0.958848,0.020576,27,9,0.000112,True
9,HC vs protan,VKS_MajRad,7.0,0.000031,-0.942387,0.028807,27,9,0.000125,True
12,HC vs deutan,TES,7.5,0.000085,-0.930556,0.034722,27,8,0.000255,True
10,HC vs protan,VKS_MinRad,16.0,0.000125,-0.868313,0.065844,27,9,0.000375,True
14,HC vs deutan,PES_BY,26.0,0.001365,-0.759259,0.120370,27,8,0.002731,True
11,HC vs protan,VKS_Angle,201.0,0.003900,0.654321,0.827160,27,9,0.007801,True
